# Section 05: 文本摘要（Summarization）核心总结

## 任务定义
**文本摘要** = 给定长文本，生成简短的摘要。同样是 Seq2Seq 任务，但语义压缩更强。

分类：
- **Extractive（抽取式）**：从原文挑选句子组合（如 baseline 的 3句摘要）
- **Abstractive（生成式）**：模型生成新词句，本节重点

## 本节任务
用 `google/mt5-small`（多语言 T5 小模型）微调：
- 输入：Amazon 英文/西班牙文书评正文（`review_body`）
- 输出：评论标题（`review_title`）—— 这是自然语言摘要的代理任务

## 与翻译（section-04）的异同
| 方面 | 翻译 | 摘要 |
|------|------|------|
| 架构 | Seq2Seq | Seq2Seq（相同）|
| 评估指标 | BLEU | ROUGE |
| 输入长度 | 短句 | 长文本（需截断）|
| 多语言 | 两种语言 | 单语言 or 多语言混合 |
| 基准对比 | 现有翻译模型 | 3句抽取式摘要 |

---
## 第一步：数据准备 — 多语言混合数据集

In [1]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

# 加载英文和西班牙文亚马逊书评
chinese_dataset = load_dataset("goosmanlei/amazon_reviews_multi", "zh")
english_dataset = load_dataset("goosmanlei/amazon_reviews_multi", "en")

# 只保留书评（book + digital_ebook_purchase），过滤其他品类
def filter_books(example):
    return example["product_category"] in ["book", "digital_ebook_purchase"]

english_books = english_dataset.filter(filter_books)
chinese_books = chinese_dataset.filter(filter_books)

# 合并英文+西班牙文，并打乱
books_dataset = DatasetDict()
for split in english_books.keys():
    books_dataset[split] = concatenate_datasets(
        [english_books[split], chinese_books[split]]
    ).shuffle(seed=42)

# 过滤标题太短的样本（标题 ≤ 2词的不适合做摘要目标）
books_dataset = books_dataset.filter(lambda x: isinstance(x["review_title"], str) and len(x["review_title"].split()) > 2)

# 查看样本结构
print("特征:", list(english_dataset["train"].features.keys()))
# ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', ...]

特征: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category']


---
## 第二步：mT5 的分词特点

mT5 使用 SentencePiece tokenizer，与 BERT 的 WordPiece 不同：
- 不需要 `as_target_tokenizer()`（mT5 的 tokenizer 对源和目标语言统一处理）
- 用 `▁` 表示词的起始（空格前缀），例如 `▁Hung er ▁Games`

In [2]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 演示 mT5 的 SentencePiece 分词
inputs = tokenizer("I loved reading the Hunger Games!")
print(tokenizer.convert_ids_to_tokens(inputs.input_ids))
# ['▁I', '▁', 'loved', '▁reading', '▁the', '▁Hung', 'er', '▁Games', '</s>']
# 注意：'▁' 表示词前的空格，</s> 是 EOS

['▁I', '▁', 'loved', '▁reading', '▁the', '▁Hung', 'er', '▁Games', '!', '</s>']


In [3]:
max_input_length = 512   # 书评可能很长
max_target_length = 30   # 标题通常很短

def preprocess_function(examples):
    """
    与翻译的预处理几乎相同，区别：
    - 输入是 review_body（长文本），目标是 review_title（短标题）
    - mT5 无需 as_target_tokenizer()
    """
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["review_title"],
        max_length=max_target_length,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = books_dataset.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(books_dataset["train"].column_names)

Map:   0%|          | 0/148 [00:00<?, ? examples/s]

---
## 第三步：评估指标 — ROUGE

**ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**：摘要任务标准指标

| 指标 | 说明 |
|------|------|
| ROUGE-1 | 1-gram（单词）重叠 |
| ROUGE-2 | 2-gram（词对）重叠 |
| ROUGE-L | 最长公共子序列（LCS），考虑语序 |
| ROUGE-Lsum | 按句子计算 LCS，再取平均 |

**ROUGE vs BLEU**：
- BLEU 侧重 Precision（预测词有多少在参考中），适合翻译
- ROUGE 侧重 Recall（参考词有多少被预测到），适合摘要

In [4]:
import evaluate
import nltk
import numpy as np

rouge_score = evaluate.load("rouge")
nltk.download("punkt_tab")
nltk.download("punkt")

# 演示 ROUGE 计算
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"
scores = rouge_score.compute(predictions=[generated_summary], references=[reference_summary])
print(f"ROUGE-1: {scores['rouge1']:.2f}")
print(f"ROUGE-2: {scores['rouge2']:.2f}")
print(f"ROUGE-L: {scores['rougeL']:.2f}")

ROUGE-1: 0.92
ROUGE-2: 0.73
ROUGE-L: 0.92


[nltk_data] Downloading package punkt_tab to /home/work/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/work/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
# 建立 Baseline：用文章前3句话作为摘要（抽取式）
from nltk.tokenize import sent_tokenize
import pandas as pd

def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])

def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text) for text in dataset["review_body"]]
    return metric.compute(predictions=summaries, references=dataset["review_title"])

score = evaluate_baseline(books_dataset["validation"], rouge_score)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = {rn: round(score[rn] * 100, 2) for rn in rouge_names}
print("3句抽取式基准:", rouge_dict)
# {'rouge1': 16.74, 'rouge2': 8.83, 'rougeL': 15.60, 'rougeLsum': 15.96}

3句抽取式基准: {'rouge1': np.float64(15.05), 'rouge2': np.float64(7.94), 'rougeL': np.float64(14.24), 'rougeLsum': np.float64(14.73)}


In [6]:
def compute_metrics(eval_pred):
    """
    ROUGE 计算时，需要在每句话后加换行符（rougeLsum 要求）
    """
    predictions, labels = eval_pred

    # 修复 predictions 中的 -100（与处理 labels 一样）
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE 期望每个句子后面有换行
    decoded_preds  = ["\n".join(sent_tokenize(pred.strip()))  for pred in decoded_preds]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]

    result = rouge_score.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    # 提取 F1 中位数，乘以 100
    result = {key: value * 100 for key, value in result.items()}
    return {k: round(v, 4) for k, v in result.items()}

---
## 第四步：训练

In [7]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

batch_size = 32
num_train_epochs = 8
logging_steps = len(tokenized_datasets["train"]) // batch_size

args = Seq2SeqTrainingArguments(
    output_dir=f"mt5-small-finetuned-amazon-en-es",
    eval_strategy="epoch",
    learning_rate=5.6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,  # 必须！摘要用 generate() 评估
    logging_steps=logging_steps,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
# 训练后 ROUGE-1 约 16.97，超过抽取式基准（16.74）

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,12.480379,4.853268,6.065600,1.414500,5.906900,5.961500
2,5.218598,3.634516,10.883500,4.275100,10.111300,10.111700
3,4.331032,3.470665,12.092200,5.632200,11.984900,11.854900
4,4.033224,3.411926,13.030800,5.809300,12.920100,12.928900
5,3.867319,3.376329,14.335000,6.670600,14.153600,14.139400
6,3.791177,3.376436,14.121100,7.379500,13.873600,14.009600
7,3.717484,3.352226,15.537600,7.882500,15.185000,15.216300
8,3.697663,3.342352,14.872500,7.796100,14.670400,14.748200


/home/work/miniforge3/envs/llm-hf/lib/python3.11/site-packages/transformers/generation/utils.py:1551: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1680, training_loss=5.13461617742266, metrics={'train_runtime': 318.7782, 'train_samples_per_second': 168.167, 'train_steps_per_second': 5.27, 'total_flos': 1.448077897070592e+16, 'train_loss': 5.13461617742266, 'epoch': 8.0})

In [ ]:
import torch
if trainer is not None:
    del trainer.model
    del trainer
    torch.cuda.empty_cache()

---
## 第五步：推理验证

In [13]:
import torch
from transformers import pipeline

hub_model_id = "goosmanlei/mt5-small-finetuned-amazon-en-es"
tokenizer = AutoTokenizer.from_pretrained(hub_model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(hub_model_id)

def summarize(text, max_new_tokens = 64):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_token=True)

# 英文评论
review_en = "Nothing special at all about this product... the book is too small and stiff and hard to write in."
title_en = "Not impressed at all... buy something else"
summary_en = summarize(review_en)
print(f"原始标题: {title_en}")
print(f"生成摘要: {summary_en}")

# 中文评论（模型支持多语言）
review_zh = "我是亚马逊的忠实顾客群中的一员，虽然某宝在中国比较流行，但是我一直认为亚马逊不会有假货，虽然东西贵一点。本着这个思想，我一直在亚马逊购物，还一直选择亚马逊自营产品购买！作为一个妈妈，母婴用品在亚马逊也是我的首选，我不知道是的对亚马逊的信任，还是我的大意，我一直没有怀疑过会有假货，也一直觉得都用的很好，直到这一次的购物。 2015年3月30日，订单编号 C03-4510442-8561649，我在亚马逊购买婴儿花王纸尿裤一包，到货后我打开包装，惊奇的发现比以前在亚马逊买的都要薄，正好上次（3月15日订单编号 C03-6109563-8124044）买的一包还剩余一片，我就拿过来比较一下，果然这次买的要薄很多，另外棉质很不均匀，有些地方整个就没有夹层，我以为不是一个供货商，检查了一下，都是东莞的同一个地方！我有觉得会不会是不同的批次，但是遗憾的是还是同一批次！（照片为证）现在这种情况只有2个可能，两包都是假货，或者其中一包是假货，我不得不承认我信任的亚马逊也有假货，我自以为比麦芽宝贝，比淘宝，比京东，比其它网站都要贵一点的亚马逊，不会有假货的亚马逊也卖了假货。 接着就进入了无比生气的维权阶段，无论我是打电话投诉，还是写邮件，没有人理你，说好的回访，也没有回访，最后的说法是让我自己找供货商，我表示我没办法找供货商，我在亚马逊买的亚马逊配送的物品，怎么找供货商？那么亚马逊在我购物过程中扮演了什么角色？最后的最后亚马逊客服给我的回话是，随便我上哪儿投诉去吧，他们是不管这个事情！！！！ 我现在还在打12315，我会一直一直打，或者我实在没有精力了，或者这个事情会有合理的解决办法，但是亚马逊，我不会再来买东西了。 最后，各位亲，我朋友从日本回来给我人肉背回花王纸尿裤，我可以担保我在这里买的每一包都是假的！ 心疼各位宝妈的那颗给宝宝最好的心，也心疼宝宝们的屁屁！"
title_zh = "真想给零颗星！亚马逊，你真让买家失望！"
summary_zh = summarize(review_zh)
print(f"\n原始标题(ES): {title_zh}")
print(f"生成摘要(ES): {summary_zh}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/

原始标题: Not impressed at all... buy something else
生成摘要: <pad> Not special at all about this product</s>


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



原始标题(ES): 真想给零颗星！亚马逊，你真让买家失望！
生成摘要(ES): <pad> 我一直没有怀疑过会有假货,但是我一直没有怀疑过会有假货</s>


---
## 总结

### 与翻译的代码差异（几乎相同的流程）

```python
# 翻译（section-04）          摘要（section-05）
源: en_sentence              源: review_body（长文本）
目标: fr_sentence            目标: review_title（短标题）
指标: BLEU                   指标: ROUGE
模型: MarianMT               模型: mT5（多语言T5）
max_input=128                max_input=512
max_target=128               max_target=30
```

### ROUGE 各子指标含义
```
ROUGE-1:   单词重叠 → 基本词汇覆盖
ROUGE-2:   词对重叠 → 短语质量
ROUGE-L:   最长公共子序列 → 整体流畅度
ROUGE-Lsum:按句 LCS → 多句摘要质量
```

### 摘要任务的特殊处理
```python
# compute_metrics 中，每句话要加 \n（rougeLsum 需要）
decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
```

### 典型 ROUGE 分数参考
| 方法 | ROUGE-1 | ROUGE-2 |
|------|---------|----------|
| 3句抽取式（基准）| 16.74 | 8.83 |
| mT5 微调后 | ~16.97 | ~8.30 |